In [ ]:
from datasets import load_from_disk
from fastdtw import fastdtw
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import csv
import os
import random
from collections import Counter
from DtwAnalyzer_noise import DTWAnalyzer
from PatternProcessor_noise import TimeSeriesPatternProcessor
from utils import DataUtils
from datetime import datetime

In [ ]:
RANDOM_SEED = 42
RESULTS_DIR = '/path/to/folder/DTW/v2/Choose_it/traffic_hourly'
DATASET_PATH = '/path/to/folder/lotsa_data/m4_weekly'

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
SAMPLE_SIZE = 80

PATTERNS = {
    '24h': {'name': '24 Hours', 'hours': 24},
    '2d': {'name': '2 Days', 'hours': 48},
    '5d': {'name': '5 Days', 'hours': 120},
    '7d': {'name': '7 Days', 'hours': 168}
}

In [ ]:
dtw_analyzer = DTWAnalyzer(PATTERNS, max_overlap_ratio=0.5)
data_utils = DataUtils(RESULTS_DIR)

ds = load_from_disk(DATASET_PATH)

total_rows = len(ds)
selected_indices = data_utils.select_random_samples(total_rows, SAMPLE_SIZE)
print(f"Selected {len(selected_indices)} rows from {total_rows} total rows for processing.")

pattern_counts = Counter({p: 0 for p in PATTERNS.keys()})
temp_data = {}

processed_count = 0
# for row_index in tqdm(range(total_rows), desc="Processing all data rows"): 
for row_index in tqdm(selected_indices, desc="Processing selected data rows"):
    sequence = ds[row_index]['target']

    min_count = min(pattern_counts.values())
    available_patterns = [p for p in PATTERNS.keys() if pattern_counts[p] == min_count]
    selected_pattern = random.choice(available_patterns)
    pattern_counts[selected_pattern] += 1

    analysis_result = dtw_analyzer.analyze_sequence(sequence, selected_pattern)
    
    if analysis_result is None:
        continue

    data_utils.collect_data(temp_data, row_index, sequence, analysis_result)
    
    processed_count += 1

csv_path = data_utils.create_csv_file()
data_utils.write_merged_csv_data(csv_path, temp_data)

print(f"Processing completed. Successfully processed {processed_count} sequences.")
print(f"Results saved to {RESULTS_DIR} folder.")
print(f"Pattern usage counts: {dict(pattern_counts)}")

#### Choose it generation

In [ ]:
current_date = datetime.now()
formatted_date = current_date.strftime('%Y%m%d')

custom_config = {
    'qa_type': 'choose it',
    'task_type': 'a multiple choice question',
    'question_template': 'Which of the following four candidate sequences, each the same length as ts0, has the most similar pattern to ts0?',
    'option': 'A) ts1. B) ts2. C) ts3. D) ts4.',
    'answer': 'A',
    'domain': 'economy',
    'freq': '1w',
}

filter_rules = [
    {'segment_type': '1st Similar', 'condition': 'less_than', 'threshold': 10000000000000000},
    # {'segment_type': '2nd Similar', 'condition': 'greater_equal', 'threshold': 0.1},
    # # {'segment_type': '3rd Similar', 'condition': 'greater_equal', 'threshold': 0.1}
]


processor = TimeSeriesPatternProcessor(verbose=True)
processor.process_and_save(
    input_csv_path="/path/to/folder/DTW/v2/Choose_it/M4_weekly/analysis_results_20250805.csv",
    dataset_path=DATASET_PATH,
    output_csv_path=f"/path/to/folder/DTW/v2/Choose_it/M4_weekly/chooseit_20250805.csv",
    data_column="target",
    output_config=custom_config,
    scale_factor= 1,            
    decimal_places= 2,          
    dtw_threshold = filter_rules,
)
success, valid_ts_ids = processor.process_data(custom_config, filter_rules)

##### with Noise

In [ ]:
current_date = datetime.now()
formatted_date = current_date.strftime('%Y%m%d')

custom_config = {
    'qa_type': 'choose it',
    'task_type': 'a multiple choice question',
    'question_template': 'Which of the following four candidate sequences, each the same length as ts0, has the most similar pattern to ts0?',
    'option': 'A) ts1. B) ts2. C) ts3. D) ts4.',
    'answer': 'A',
    'domain': 'economy',
    'freq': '1w',
}
processor = TimeSeriesPatternProcessor(verbose=True)

filter_rules = [
    {'segment_type': '1st Similar', 'condition': 'less_than', 'threshold': 10000000000000000},
    # {'segment_type': '2nd Similar', 'condition': 'greater_equal', 'threshold': 0.1},
    # # {'segment_type': '3rd Similar', 'condition': 'greater_equal', 'threshold': 0.1}
]

processor.process_and_save(
    input_csv_path="/path/to/folder/DTW/v2/Choose_it/traffic_hourly/analysis_results_20250805.csv",
    dataset_path=DATASET_PATH,
    output_csv_path=f"/path/to/folder/DTW/v2/Choose_it/traffic_hourly/chooseit_20250805.csv",
    data_column="target",
    output_config=custom_config, 
    scale_factor= 100,            
    decimal_places= 2,            
    dtw_threshold = filter_rules, 
    add_noise=True,
    noise_type='gaussian',
    noise_level= 0.2
)
success, valid_ts_ids = processor.process_data(custom_config, filter_rules)

#### Find it generation

In [ ]:
from FinditProcessor import extend_option_index_and_update_value
csv_file_path = "/path/to/folder/DTW/v2/Choose_it/traffic_hourly/chooseit_53.csv"
dataset_path = DATASET_PATH
output_file_path = "/path/to/folder/DTW/v2/Find_it/traffic_hourly/findit_53.csv"

processed_df = extend_option_index_and_update_value(
    csv_file_path, 
    dataset_path, 
    output_file_path,
    scale_factor=100.0,
    decimal_places=2,
    add_noise=True,
    noise_level=0.5,
    noise_type='gaussian'
)

### Reverse it generation

In [ ]:
from ReverseitProcessor import process_csv_reverse
CSV_FILE_PATH = "/path/to/folder/DTW/v2/Choose_it/traffic_hourly/chooseit_50.csv"
OUTPUT_PATH = "/path/to/folder/DTW/v2/Choose_it/traffic_hourly/reverseit_20250805.csv"

processed_df = process_csv_reverse(CSV_FILE_PATH, OUTPUT_PATH)

### Smooth it generation

In [ ]:
from SmoothitProcessor import smooth_csv_data
CSV_FILE_PATH = "/path/to/folder/DTW/v2/Choose_it/traffic_hourly/chooseit_50.csv"
OUTPUT_PATH = "/path/to/folder/DTW/v2/Smooth_it/traffic_hourly/smoothit_20250805.csv"

smooth_csv_data(CSV_FILE_PATH, OUTPUT_PATH, window_size=3)

#### Visualization

In [ ]:
data_utils = DataUtils(results_dir="/nas/datasets/AAA_TS_Benchmark/DTW/v2/Find_it/traffic_hourly")
data_utils.plot_all_rows_separately("/nas/datasets/AAA_TS_Benchmark/DTW/v2/Choose_it/traffic_hourly/chooseit_53.csv")

# data_utils = DataUtils(results_dir="/nas/datasets/AAA_TS_Benchmark/DTW/find_it/traffic", selected_mode=SELECTED_MODE)
# data_utils.plot_all_rows_separately("/nas/datasets/AAA_TS_Benchmark/DTW/find_it/traffic/qa_final/Findit_exam_qa_50.csv")

In [ ]:
dtw_analyzer = DTWAnalyzer(PATTERNS)
result_df = dtw_analyzer.process_csv_file('/nas/datasets/AAA_TS_Benchmark/MutiTS/DTW/find_it/m4_findit_50.csv', '/nas/datasets/AAA_TS_Benchmark/MutiTS/DTW/image/m4_findit_50_review.csv')

In [ ]:
from image_analyze import analyze_all_csv_rows
input_dir ="/nas/datasets/AAA_TS_Benchmark/MutiTS/DTW/image/m4_findit_50_review.csv"
output_dir="/nas/datasets/AAA_TS_Benchmark/MutiTS/DTW/image/image_analyze"
analyze_all_csv_rows(input_dir,output_dir)

#### Merge

In [ ]:
import os
import glob
import pandas as pd

def merge_csv_files(input_dir, output_prefix='Smoothit_exam_qa'):
    csv_files = glob.glob(os.path.join(input_dir, '*.csv'))
    
    if not csv_files:
        print(f"No CSV files found in {input_dir} directory")
        return
    
    all_data = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            all_data.append(df)
            print(f"Read: {file}")
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    merged_df = pd.concat(all_data, ignore_index=True)
    
    if 'num' in merged_df.columns:
        merged_df['num'] = range(1, len(merged_df) + 1)
    
    total_rows = len(merged_df)
    
    output_filename = f"{output_prefix}_{total_rows}.csv"
    output_path = os.path.join(input_dir, output_filename)
    
    merged_df.to_csv(output_path, index=False)
    print(f"Merge completed, total {total_rows} rows of data, saved to: {output_path}")
    print(f"num column has been renumbered, range from 1 to {total_rows}")

if __name__ == "__main__":
    directory = f"/path/to/folder/Real-world dataset/DTW_datasets/final"
    merge_csv_files(directory)

#### Shuffle

In [ ]:
import pandas as pd
import ast
import random

def rebalance_answers_with_option_shuffle(csv_file_path, output_file_path=None):
    df = pd.read_csv(csv_file_path)
    
    if output_file_path is None:
        output_file_path = csv_file_path.replace('.csv', '_rebalanced.csv')
    
    answer_options = ['A', 'B', 'C', 'D']
    total_rows = len(df)
    
    print("Original answer distribution:")
    print(df['answer'].value_counts().sort_index())
    
    answer_distribution = []
    for i in range(total_rows):
        answer_distribution.append(answer_options[i % 4])
    
    random.shuffle(answer_distribution)
    
    for idx, row in df.iterrows():
        current_answer = row['answer']  
        new_answer = answer_distribution[idx]  
        
        if current_answer == new_answer:
            continue
            
        try:
            value_list = ast.literal_eval(row['value'])
            
            if len(value_list) >= 5:
                ts0 = value_list[0]
                ts_options = value_list[1:5]

                current_answer_index = answer_options.index(current_answer)
                new_answer_index = answer_options.index(new_answer)

                new_ts_options = ts_options.copy()
                new_ts_options[new_answer_index] = ts_options[current_answer_index]
                new_ts_options[current_answer_index] = ts_options[new_answer_index]

                new_value_list = [ts0] + new_ts_options
                if len(value_list) > 5:
                    new_value_list.extend(value_list[5:])
                
                df.at[idx, 'value'] = str(new_value_list)
                df.at[idx, 'answer'] = new_answer
                
        except Exception as e:
            print(f"Error processing row {idx+1}: {e}")
            continue
    
    df.to_csv(output_file_path, index=False)
    print(f"\nProcessing completed! Results saved to: {output_file_path}")
    print("Balanced answer distribution statistics:")
    print(df['answer'].value_counts().sort_index())
    
    return df

if __name__ == "__main__":
    input_file = "/path/to/folder/test dataset/caption_qa_120.csv"
    df = rebalance_answers_with_option_shuffle(input_file)

In [ ]:
import pandas as pd

def delete_csv_rows_by_numbers(input_file, output_file, num_delete):
    try:
        df = pd.read_csv(input_file)
        
        print(f"Original data information:")
        print(f"  - Total rows: {len(df)}")
        print(f"  - Column names: {list(df.columns)}")
        delete_indices = [row_num - 1 for row_num in num_delete]
        valid_delete_indices = [idx for idx in delete_indices if 0 <= idx < len(df)]
        invalid_rows = [num_delete[i] for i, idx in enumerate(delete_indices) if idx not in valid_delete_indices]
        
        if invalid_rows:
            print(f"Warning: The following row numbers are out of data range and will be ignored: {invalid_rows}")

        filtered_df = df.drop(index=valid_delete_indices).reset_index(drop=True)

        if 'num' in filtered_df.columns:
            filtered_df['num'] = range(1, len(filtered_df) + 1)
            print(f"Renumbered num column (from 1 to {len(filtered_df)})")
        else:
            print("Warning: 'num' column not found, skipping renumbering")

        filtered_df.to_csv(output_file, index=False)
        
        print(f"\nDeletion completed!")
        print(f"Original data rows: {len(df)}")
        print(f"Deleted rows: {len(valid_delete_indices)}")
        print(f"Remaining rows: {len(filtered_df)}")
        print(f"Results saved to: {output_file}")
        
        return True
        
    except FileNotFoundError:
        print(f"Error: File not found {input_file}")
        return False
    except Exception as e:
        print(f"Error: {str(e)}")
        return False
    
if __name__ == "__main__":
    num_delete = [14,48,52]
    
    input_file = "/nas/datasets/AAA_TS_Benchmark/DTW/v2/Find_it/traffic_hourly/findit_53.csv"
    output_file = "/nas/datasets/AAA_TS_Benchmark/DTW/v2/Find_it/traffic_hourly/findit_50.csv"
    
    delete_csv_rows_by_numbers(input_file, output_file, num_delete)